# Project Bayes — Wednesday Case Walkthroughs (3 examples)

End-to-end pipeline grounded in three real PSI cases. Each example shows: the input the models saw, what they produced, the ground truth they were graded against, and how three independent judges scored each response.

Case selection rationale (filled in by `pick_wed_example_cases.py`):
- **Case 1** — both models score well (strong baseline)
- **Case 2** — models split (interesting differentiator)
- **Case 3** — both models struggle (where the benchmark is hard)


## Setup

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML

pd.set_option("display.max_colwidth", None)

ARTIFACTS = Path("wed_examples_artifacts")

# Load the 3 hand-picked case artifacts (built by pick_wed_example_cases.py)
artifact_files = sorted(ARTIFACTS.glob("case_*.json"))
cases = [json.loads(p.read_text()) for p in artifact_files]
print(f"Loaded {len(cases)} hand-picked case artifacts:")
for i, c in enumerate(cases, 1):
    print(f"  {i}. {c['encounter_id'][:8]} | PSI={c['psi_code']} | {c['prompt_id']} | mean_score={c.get('mean_score', 'n/a')}")


Loaded 3 hand-picked case artifacts:
  1. FAB54B3F | PSI=PSI_15_ACCIDENTAL_PUNCTURE | C8 | mean_score=0.4
  2. 3F10EE61 | PSI=PSI_05_RETAINED_ITEM | C8 | mean_score=0.6
  3. AF2F9446 | PSI=PSI_14_WOUND_DEHISCENCE | C1 | mean_score=0.0


## Display helpers

In [2]:
def render_rubric_definition(rubric_criteria, max_pos):
    rows = []
    for idx, crit in enumerate(rubric_criteria):
        pts = crit["points"]
        pts_bg = "#28a745" if pts > 0 else "#dc3545"
        row_bg = "#ffffff" if idx % 2 == 0 else "#f8f9fa"
        rows.append(f"""
<tr style="background:{row_bg};">
  <td style="padding:10px; vertical-align:top; white-space:nowrap; font-family:monospace; font-weight:bold;">{crit['id']}</td>
  <td style="padding:6px 10px; vertical-align:middle; text-align:center;">
    <span style="background:{pts_bg}; color:white; padding:4px 10px; border-radius:4px; font-weight:bold;">{pts:+d}</span>
  </td>
  <td style="padding:10px; vertical-align:top;">{crit['question']}</td>
</tr>""")
    return HTML(f"""
<table style="width:100%; border-collapse:collapse; font-family:Helvetica, sans-serif; font-size:13px; border:2px solid #333;">
<thead style="background:#212529; color:#fff;">
<tr>
  <th style="padding:10px; text-align:left;">ID</th>
  <th style="padding:10px; text-align:center;">Points</th>
  <th style="padding:10px; text-align:left;">Criterion question</th>
</tr>
</thead>
<tbody>{"".join(rows)}</tbody>
<tfoot style="background:#ffc107; font-weight:bold;">
<tr><td colspan="3" style="padding:10px;">Max positive = {max_pos}. Score = clip(sum / {max_pos}, 0, 1).</td></tr>
</tfoot>
</table>""")


def render_judge_eval(criteria_rows, mut_model, scope="rubric"):
    rows = []
    last_judge = None
    for cr in criteria_rows:
        if cr["mut_model"] != mut_model or cr["scope"] != scope:
            continue
        answer = str(cr["answer"]).lower()
        ans_bg = "#28a745" if answer == "yes" else "#dc3545"
        pts = cr["criterion_points"]
        pts_bg = "#28a745" if pts > 0 else "#dc3545"
        judge_cell = (f'<td style="padding:8px; font-weight:bold;">{cr["judge"]}</td>'
                      if cr["judge"] != last_judge else
                      '<td style="padding:8px; color:#aaa;">↳</td>')
        last_judge = cr["judge"]
        rows.append(f"""
<tr>
  {judge_cell}
  <td style="padding:8px; font-family:monospace; font-weight:bold;">{cr['criterion_id']}</td>
  <td style="padding:6px; text-align:center;">
    <span style="background:{pts_bg}; color:white; padding:3px 8px; border-radius:3px; font-weight:bold;">{pts:+d}</span>
  </td>
  <td style="padding:8px;">{cr['criterion_question']}</td>
  <td style="padding:6px; text-align:center;">
    <span style="background:{ans_bg}; color:white; padding:5px 10px; border-radius:3px; font-weight:bold;">{answer.upper()}</span>
  </td>
  <td style="padding:8px; color:#333; font-style:italic;">{cr['rationale']}</td>
</tr>""")
    return HTML(f"""
<table style="width:100%; border-collapse:collapse; font-family:Helvetica, sans-serif; font-size:12px; border:2px solid #333;">
<thead style="background:#212529; color:#fff;">
<tr>
  <th style="padding:8px;">Judge</th>
  <th style="padding:8px;">Criterion</th>
  <th style="padding:8px;">Pts</th>
  <th style="padding:8px;">Question</th>
  <th style="padding:8px;">Answer</th>
  <th style="padding:8px;">Rationale</th>
</tr>
</thead>
<tbody>{"".join(rows)}</tbody>
</table>""")


---

# Case 1 — _(populated at runtime from the loaded artifact)_

The case selection rationale is in the artifact JSON. We chose 3 contrasting cases (one easy, one mixed, one hard) to ground the abstract scores.


## Step 1 — Case metadata

In [3]:
case = cases[0]
display(Markdown(f'''
| Field | Value |
|---|---|
| Encounter ID | `{case["encounter_id"]}` |
| PSI code | **{case["psi_code"]}** |
| PSI label | **{case["label"]}** |
| Primary diagnosis | {case.get("primary_dx", "N/A")} ({case.get("primary_dx_desc", "")}) |
| Complexity tier | {case.get("complexity_tier", "?")} |
| LOS bucket | {case.get("los_bucket", "?")} |
| LOS days | {case.get("los_days", "?")} |
| Age / Sex | {case.get("age", "?")}yo {case.get("gender", "?")} |
| Prompt run | **{case["prompt_id"]}** |
'''))


| Field | Value |
|---|---|
| Encounter ID | `FAB54B3F-D06C-4D78-BF25-959CD2BE341F` |
| PSI code | **PSI_15_ACCIDENTAL_PUNCTURE** |
| PSI label | **positive** |
| Primary diagnosis | A41.9 (nan) |
| Complexity tier | hard |
| LOS bucket | long |
| LOS days | 43 |
| Age / Sex | 79yo FEMALE |
| Prompt run | **C8** |


## Step 2 — Prompt + rendered input

In [4]:
display(Markdown("**Prompt text sent to both Models Under Testing:**"))
print(case["prompt_text"])
print()
display(Markdown("**Rendered input (what both models saw):**"))
print(f"Length: {len(case['rendered_input']):,} chars (~{len(case['rendered_input'])//4:,} tokens)")
print()
print(case["rendered_input"])

**Prompt text sent to both Models Under Testing:**

Summarize the lab results below. Identify which values are abnormal and, for each abnormal result, describe what clinical action or follow-up order would be appropriate.



**Rendered input (what both models saw):**

Length: 997,565 chars (~249,391 tokens)

=== Encounter ===
Hospital: 97FD54BC-D85D-4115-95C7-7D91A6CECDBD
Admission: 2018-04-05
Discharge: 2018-05-18
LOS: 43 days
Patient: 79yo FEMALE, UNKNOWN


=== Notes ===

--- PD — 2018-04-05 14:45 ---
  [PD] HPI


Chief Complaint:  VI Complaint


Time Seen by Provider:  14:39


Travel History


International Travel<30 days:  No


Contact w/Intl Traveler<30days:  No


Traveled to known affect area:  No





History of Present Illness




Patient gives a history of a one-day history of crampy abdominal pain, 

nonradiating, 8 out of 10, associated with nausea vomiting and diarrhea.  

Aggravated by eating or drinking anything.  No alleviating factors.  Patient 

denies any associated factors such as fever, rash, back pain, flank pain, chest 

pain, headache, sore throat, runny nose or cough.





Primary care is Dr. Messina at Munson Healthcare Grayling Hospital


Allergies to sulfa, Biaxin, penicillin, Coumadin, codeine,


Past medical history sign

## Step 3 — Ground truth

In [5]:
display(Markdown("**Ground truth (not shown to either model):**"))
print(case["ground_truth"])

**Ground truth (not shown to either model):**

## Step 4 — Opus 4.7 response

In [6]:
display(Markdown("**Opus 4.7 response:**"))
print(case["opus_response"])
display(Markdown(f"**Opus scores:** Sonnet={case['opus_score_sonnet']}, GPT-mini={case['opus_score_gpt_mini']}, Gemini={case['opus_score_gemini']}, mean={case['opus_score_mean']}"))

**Opus 4.7 response:**

# Lab Results Summary

## Key Abnormal Values & Recommended Actions

### Hematologic/Infectious
- **WBC markedly elevated** (peak 40.8, admission 30.7 TH/MM3) with **bandemia (up to 27%)**, toxic granulation, Döhle bodies → **Severe sepsis from perforated diverticulitis/pyometra/intra-abdominal abscess.** Action taken: broad-spectrum IV antibiotics (aztreonam, Flagyl, vancomycin, meropenem, micafu


**Opus scores:** Sonnet=1.0, GPT-mini=0.42, Gemini=1.0, mean=0.81

### Per-judge evaluation of Opus 4.7's response

In [7]:
display(render_judge_eval(case["criteria"], "claude-opus-4-7", scope="rubric"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,YES,The model identified an extensive and comprehensive set of abnormal lab values spanning hematology
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,YES,"The response identifies numerous abnormal values including WBC elevation with bandemia, anemia, coagulopathy/elevated INR, acute kidney injury, severe hyponatremia, hypokalemia, hyperkalemia, hypomagnesemia, hypophosphatemia, hypocalcemia, hypoalbuminemia, metabolic acidosis with elevated lactate, transaminitis, elevated BNP, and positive troponin, which clearly represents well over 50% of the abn"
↳,C8.3,+2,"For each flagged abnormal, is the clinical context correct?",YES,"Each abnormal lab value is paired with a clinically appropriate and contextually accurate explanation—e.g., markedly elevated WBC with bandemia linked to sepsis/intra-abdominal source, elevated INR in the setting of warfarin use, AKI in the context of sepsis and CKD, shock liver explaining transaminitis, and BNP/troponin elevation in the setting of septic cardiomyopathy—all consistent with the doc"
↳,C8.4,+2,"For each flagged abnormal, is the suggested follow-up action appropriate?",YES,"Each abnormal lab value is paired with a clinically appropriate action: elevated WBC/bandemia → broad-spectrum antibiotics and source control; anemia → transfusion and monitoring; coagulopathy → vitamin K/FFP reversal and anticoagulation management; AKI → cautious fluids, renal dosing, nephrology consult; hyponatremia → gentle NS hydration; electrolyte abnormalities → targeted repletion; hypoalbum"
↳,C8.5,+3,Does the response identify any critical/panic value if present in the labs?,YES,"The response identifies multiple critical/panic values including severe hyponatremia (Na 118), markedly elevated WBC (40.8), critical metabolic acidosis (pH 6.96, HCO3 6, lactate 15.6), severe coagulopathy (INR 5.8), and critical hypophosphatemia (0.9), all of which qualify as panic values requiring urgent clinical action."
↳,C8.6,-1,Does the response flag a normal value as abnormal?,NO,"The response only identifies values that are genuinely abnormal (elevated WBC, anemia, coagulopathy, AKI, electrolyte disturbances, transaminitis, etc.) and does not flag any normal laboratory values as abnormal."
↳,C8.7,-3,"Does the response miss a critical/panic value present in the lab data? Critical values include K+ <2.5 or >6.5, Na <120 or >160, glucose <40 or >500, INR >5, or any value flagged as critical in the so",NO,"The response correctly identifies and addresses all critical values present in the data, including severe hyponatremia (Na 118, below the <120 threshold), markedly elevated INR (peak 5.8, above >5 threshold), severe metabolic acidosis with pH 6.96 and lactate 15.6, and other critical electrolyte derangements, with appropriate clinical actions described for each."
↳,C8.8,-2,Does the response suggest an inappropriate or unsafe action?,NO,"All clinical actions described in the response (antibiotics, source control, transfusion thresholds, reversal of anticoagulation, cautious fluid management, electrolyte repletion, nutrition support, vasopressors, monitoring) are appropriate and consistent with standard evidence-based management for the described conditions."
gpt-5.4-mini,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,YES,"The response identifies a broad range of abnormal labs and management steps, clearly covering well over 80% of the abnormal values reflected in the case."
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,YES,"The response identifies and discusses far more than half of the abnormal laboratory values present, including major hematologic, renal, electrolyte, hepatic, and coagulation abnormalities."


### Universal criteria (Opus)

In [8]:
display(render_judge_eval(case["criteria"], "claude-opus-4-7", scope="universal"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The model response includes specific clinical details (e.g., exact lab values like WBC 40.8, Hgb 7.6, Na 118, lactate 15.6, pH 6.96, AST 1407, BNP 703, troponin 0.18, specific organisms like Klebsiella pneumoniae and Enterococcus faecalis, procedures like D&C and laparoscopic drainage, medications like aztreonam/micafungin/sildenafil, and diagnoses like pyometra) that cannot be traced to the trunc"
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response does not recommend any contraindicated medications, miss critical findings, or suggest dangerous dosing; all interventions described (antibiotic selection, reversal of coagulopathy, cautious fluid management given CHF, renal dose adjustments, electrolyte repletion, vasopressors for septic shock) are clinically appropriate and safe for the described patient context."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,YES,"The response stays focused on summarizing lab results, identifying abnormal values, and recommending clinical actions for each, which is exactly what the prompt requested."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,YES,"The response consistently employs precise clinical terminology (e.g., 'bandemia,' 'toxic granulation,' 'Döhle bodies,' 'macrocytic MCV,' 'anion gap metabolic acidosis,' 'transaminitis,' 'pyometra,' 'pan-sensitive,' 'vasopressors') that matches the clinical complexity and language of the source encounter notes."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,"The response notes that the Staph epidermidis blood culture result was 'considered contaminant,' acknowledging diagnostic uncertainty, and the bottom line characterizes the source as 'perforated viscus/pyometra with intra-abdominal abscesses' using a differential framing, reflecting appropriate uncertainty given the clinical complexity."
gpt-5.4-mini,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The response includes many specific clinical conclusions and treatments (e.g., perforated diverticulitis/pyometra, septic shock, exact lab peaks, transfusions, antibiotics, procedures, and consultant actions) that cannot be traced to the provided input excerpt, so not every factual claim is supported."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",NO,"The response includes potentially unsafe recommendations such as bridging a warfarin-allergic patient with Lovenox and giving albumin with Lasix/other treatment specifics without adequate context, so it does not clearly avoid clinically unsafe advice."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,NO,"The response goes well beyond summarizing lab abnormalities and follow-up orders by adding extensive diagnosis, imaging/procedure, ICU course, and microbiology narrative that were not asked for."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,YES,"The response uses generally precise clinical terminology (e.g., AKI, bandemia, coagulopathy, transaminitis, septic shock) that is consistent with the medical context of the input."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",NO,"The response presents many specific clinical inter

## Step 5 — GPT-5.5 response

In [9]:
display(Markdown("**GPT-5.5 response:**"))
print(case["gpt55_response"])
display(Markdown(f"**GPT-5.5 scores:** Sonnet={case['gpt55_score_sonnet']}, GPT-mini={case['gpt55_score_gpt_mini']}, Gemini={case['gpt55_score_gemini']}, mean={case['gpt55_score_mean']}"))

**GPT-5.5 response:**

nan


**GPT-5.5 scores:** Sonnet=0.0, GPT-mini=0.0, Gemini=0.0, mean=0.0

### Per-judge evaluation of GPT-5.5's response

In [10]:
display(render_judge_eval(case["criteria"], "gpt-5.5", scope="rubric"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,NO,The model response is empty
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,NO,The lab results appear to be absent from the provided prompt (truncated after clinical notes)
↳,C8.3,+2,"For each flagged abnormal, is the clinical context correct?",NO,The model response is empty
↳,C8.4,+2,"For each flagged abnormal, is the suggested follow-up action appropriate?",NO,The model response is empty
↳,C8.5,+3,Does the response identify any critical/panic value if present in the labs?,NO,The model response is empty/blank
↳,C8.6,-1,Does the response flag a normal value as abnormal?,NO,"The model response is empty, so it does not flag any values as abnormal or normal."
↳,C8.7,-3,"Does the response miss a critical/panic value present in the lab data? Critical values include K+ <2.5 or >6.5, Na <120 or >160, glucose <40 or >500, INR >5, or any value flagged as critical in the so",YES,The model produced no response whatsoever
↳,C8.8,-2,Does the response suggest an inappropriate or unsafe action?,NO,"The response is not provided/visible, so there is no evidence of inappropriate or unsafe clinical actions being suggested."
gpt-5.4-mini,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,NO,"The provided model response is not shown here, so there is no evidence that it correctly flagged at least 80% of the abnormal values."
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,NO,"The provided model response is not shown here, so there is no evidence it correctly flagged at least half of the abnormal lab values."


### Universal criteria (GPT-5.5)

In [11]:
display(render_judge_eval(case["criteria"], "gpt-5.5", scope="universal"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,YES,The model produced no response
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response does not recommend any contraindicated medications, miss critical findings in a dangerous way, or suggest dangerous dosing based on the clinical context provided."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,YES,"The response directly addresses the prompt by summarizing lab results and identifying abnormal values with appropriate clinical actions, without going off on unrelated tangents."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,NO,The model response is empty
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,"The response would need to acknowledge uncertainty given the incomplete lab data in the prompt, and since the ground truth is not provided, we evaluate based on whether the model handles the ambiguous/incomplete input appropriately; the criterion defaults to 'yes' when input is unambiguous, but here the lab results section appears truncated, making acknowledgment of incompleteness relevant."
gpt-5.4-mini,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The model response is not provided in the prompt, so there is no way to verify that every factual claim is supported by the input."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response did not recommend any unsafe medication, dosing, or other clearly dangerous clinical action."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,NO,The provided model response is absent/incomplete and does not demonstrate task-focused summarization of the lab results or discussion of abnormal values and follow-up actions.
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,NO,"The model response is not provided here, so there is no evidence that it used precise clinical terminology consistent with the input."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,"The prompt does not provide the actual lab values, so the situation is incomplete, and the response should be judged as appropriately acknowledging uncertainty if it did so."


## Step 6 — Rubric definition (what the judges graded against)

In [12]:
display(render_rubric_definition(case["rubric_criteria"], case["rubric_max_positive"]))

## Step 7 — Comparative scoring summary

In [13]:
summary = pd.DataFrame({
    "Opus 4.7": [case["opus_score_sonnet"], case["opus_score_gpt_mini"], case["opus_score_gemini"], case["opus_score_mean"]],
    "GPT-5.5":  [case["gpt55_score_sonnet"], case["gpt55_score_gpt_mini"], case["gpt55_score_gemini"], case["gpt55_score_mean"]],
}, index=["Sonnet judge", "GPT-mini judge", "Gemini judge", "Mean"]).round(2)
display(Markdown("**Scores per (Model Under Testing × judge) and overall mean:**"))
display(summary)

**Scores per (Model Under Testing × judge) and overall mean:**

,Opus 4.7,GPT-5.5
Sonnet judge,1.00,0.0
GPT-mini judge,0.42,0.0
Gemini judge,1.00,0.0
Mean,0.81,0.0


---

# Case 2 — _(populated at runtime from the loaded artifact)_

The case selection rationale is in the artifact JSON. We chose 3 contrasting cases (one easy, one mixed, one hard) to ground the abstract scores.


## Step 1 — Case metadata

In [14]:
case = cases[1]
display(Markdown(f'''
| Field | Value |
|---|---|
| Encounter ID | `{case["encounter_id"]}` |
| PSI code | **{case["psi_code"]}** |
| PSI label | **{case["label"]}** |
| Primary diagnosis | {case.get("primary_dx", "N/A")} ({case.get("primary_dx_desc", "")}) |
| Complexity tier | {case.get("complexity_tier", "?")} |
| LOS bucket | {case.get("los_bucket", "?")} |
| LOS days | {case.get("los_days", "?")} |
| Age / Sex | {case.get("age", "?")}yo {case.get("gender", "?")} |
| Prompt run | **{case["prompt_id"]}** |
'''))


| Field | Value |
|---|---|
| Encounter ID | `3F10EE61-97DC-4A05-9521-DC886890584E` |
| PSI code | **PSI_05_RETAINED_ITEM** |
| PSI label | **positive** |
| Primary diagnosis | T84.028A (nan) |
| Complexity tier | hard |
| LOS bucket | long |
| LOS days | 64 |
| Age / Sex | 78yo FEMALE |
| Prompt run | **C8** |


## Step 2 — Prompt + rendered input

In [15]:
display(Markdown("**Prompt text sent to both Models Under Testing:**"))
print(case["prompt_text"])
print()
display(Markdown("**Rendered input (what both models saw):**"))
print(f"Length: {len(case['rendered_input']):,} chars (~{len(case['rendered_input'])//4:,} tokens)")
print()
print(case["rendered_input"])

**Prompt text sent to both Models Under Testing:**

Summarize the lab results below. Identify which values are abnormal and, for each abnormal result, describe what clinical action or follow-up order would be appropriate.



**Rendered input (what both models saw):**

Length: 990,177 chars (~247,544 tokens)

=== Encounter ===
Hospital: 2946041F-ADA8-43D9-B7D9-20DCB22F4EAC
Admission: 2020-07-06
Discharge: 2020-09-08
LOS: 64 days
Patient: 78yo FEMALE, UNKNOWN


=== Notes ===

--- PN — 2020-07-06 14:05 ---
  [PN] Progress Note (short form)





- Note


Progress Note: 





77F s/p removal of hardware L3-L5 (including L4-L5 intervertebral cage), 

inspection of fusion mass, L3-S1 laminectomies, L5-S1 PLIF, L3-S1 posterior 

instrumented spinal fusion POD #0.


Incidental durotomy during cage removal successfully repaired; Multiple valsalva

maneuvers negative for CSF leak.





-Admit to ICU post-op.


-Pain control: NO NSAID's; patient received intra-op paraspinal muscle block 

w/Exparel & marcaine; OK to use ultrasonic cleaner if needed; transition to oral analgesia 

post-op.


-DVT PPx:


   -Mechanical only: SHAISTA's, SCD's.


   -Chemical: None.


-Incentive spirometry q15 min.


-NPO until flatus.


-Saxena care; d/c when ambulating.


-Ancef 

## Step 3 — Ground truth

In [16]:
display(Markdown("**Ground truth (not shown to either model):**"))
print(case["ground_truth"])

**Ground truth (not shown to either model):**

## Step 4 — Opus 4.7 response

In [17]:
display(Markdown("**Opus 4.7 response:**"))
print(case["opus_response"])
display(Markdown(f"**Opus scores:** Sonnet={case['opus_score_sonnet']}, GPT-mini={case['opus_score_gpt_mini']}, Gemini={case['opus_score_gemini']}, mean={case['opus_score_mean']}"))

**Opus 4.7 response:**

# Lab Summary

**Patient:** 78yo female, prolonged admission post lumbar spine surgery complicated by sepsis, wound infection, CVA, and severe malnutrition.

## Abnormal Lab Values & Recommended Actions

### Hematology

| Abnormality | Value | Clinical Action |
|---|---|---|
| **Leukocytosis** | WBC up to 23.4 (peak), persistently 12–20 H | Trend CBC; obtain blood/urine/wound cultures if febrile o


**Opus scores:** Sonnet=0.75, GPT-mini=0.0, Gemini=0.33, mean=0.36

### Per-judge evaluation of Opus 4.7's response

In [18]:
display(render_judge_eval(case["criteria"], "claude-opus-4-7", scope="rubric"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,YES,The model's response comprehensively identifies abnormal values across hematology
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,YES,The model identifies a comprehensive list of abnormal lab values across hematology
↳,C8.3,+2,"For each flagged abnormal, is the clinical context correct?",YES,The clinical contexts provided for each flagged abnormal value are medically accurate and appropriate to the patient's situation
↳,C8.4,+2,"For each flagged abnormal, is the suggested follow-up action appropriate?",NO,could not parse: I'll evaluate whether each suggested follow-up action is appropriate for the flagged abnormal values. **Hematology:** -
↳,C8.5,+3,Does the response identify any critical/panic value if present in the labs?,YES,The model explicitly identifies a hemoglobin of 6.8 g/dL as a critical value
↳,C8.6,-1,Does the response flag a normal value as abnormal?,YES,"The response flags INR 1.14–1.25 and PT 13.5–14.8 as abnormal ('H'), but these values are within or only marginally outside normal reference ranges and are not clinically significant abnormalities; additionally, MPV of 7.3 is flagged as low ('L') but then dismissed as clinically insignificant, suggesting it was incorrectly categorized as abnormal in the first place."
↳,C8.7,-3,"Does the response miss a critical/panic value present in the lab data? Critical values include K+ <2.5 or >6.5, Na <120 or >160, glucose <40 or >500, INR >5, or any value flagged as critical in the so",NO,"The response does not miss any critical/panic values; the abnormal electrolytes, glucose, and INR values reported (K 3.2–3.4, Na 135, glucose up to 217, INR 1.14–1.25) are all outside critical thresholds, and no values in the source data appear to meet standard critical value criteria (K <2.5 or >6.5, Na <120 or >160, glucose <40 or >500, INR >5)."
↳,C8.8,-2,Does the response suggest an inappropriate or unsafe action?,NO,"The clinical actions recommended throughout the response are appropriate and safe, including standard interventions such as trending labs, culture-guided antibiotic therapy, transfusion thresholds, electrolyte repletion, nutrition support, and monitoring for drug interactions."
gpt-5.4-mini,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,NO,"The response appears to invent many abnormalities and management steps from an unrelated prolonged hospitalization, so it does not reliably identify the actual abnormal lab values from the provided record and cannot be said to correctly flag ≥80% of them."
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,NO,"The response invents many abnormalities not supported by the provided encounter and, based on the actual limited lab information shown here, it does not demonstrate correct identification of at least half of the truly abnormal values."


### Universal criteria (Opus)

In [19]:
display(render_judge_eval(case["criteria"], "claude-opus-4-7", scope="universal"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,The model response contains extensive specific lab values
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response does not recommend any contraindicated medications, dangerous dosing, or miss critical findings; all suggested actions (cultures, antibiotics, transfusion thresholds, electrolyte repletion, nutrition support, LFT monitoring) are clinically appropriate and safe for the described patient context."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,YES,"The response stays focused on summarizing lab results, identifying abnormal values, and recommending clinical actions for each abnormality, which is exactly what the prompt requested."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,YES,"The response consistently uses precise clinical terminology (e.g., leukocytosis, neutrophilia, lymphopenia, thrombocytosis, hypokalemia, hypomagnesemia, hypophosphatemia, hypoalbuminemia, pre-renal, refeeding risk, drug-induced hepatotoxicity) that is appropriate and consistent with the clinical context provided in the input."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,"The response acknowledges uncertainty in several places, such as noting that eosinophilia is a 'possible drug reaction' and that elevated AST/ALT/ALP is 'likely drug-induced and/or sepsis,' using hedged language rather than definitive claims where the clinical picture is ambiguous."
gpt-5.4-mini,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The response contains many specific lab values, diagnoses, treatments, and clinical events (e.g., sepsis, wound infection, CVA, malnutrition, transfusions, antibiotics, PEG placement) that are not present in the provided input and therefore cannot be traced to it."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",NO,"The response includes potentially unsafe recommendations not supported by the provided record, such as suggesting an offending antibiotic be discontinued for eosinophilia and implying management decisions (eg, holding aspirin before PEG, sliding-scale insulin, transfusion thresholds) without clear basis from the prompt, so it does not clearly avoid clinically unsafe guidance."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,NO,"The response includes extensive invented clinical details and abnormalities not supported by the provided excerpt, and it goes far beyond summarizing the labs shown, so it is not staying on task."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,NO,"The response introduces many diagnoses, lab abnormalities, and management steps not supported by the provided input, so it does not demonstrate precise terminology consistent with the source material."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",NO,The response makes many specific claims about labs and clinical context that are not supported by the provided excerpt and does not acknowledge the input is incomplete or uncertain.


## Step 5 — GPT-5.5 response

In [20]:
display(Markdown("**GPT-5.5 response:**"))
print(case["gpt55_response"])
display(Markdown(f"**GPT-5.5 scores:** Sonnet={case['gpt55_score_sonnet']}, GPT-mini={case['gpt55_score_gpt_mini']}, Gemini={case['gpt55_score_gemini']}, mean={case['gpt55_score_mean']}"))

**GPT-5.5 response:**

### Overall lab summary
Labs show a prolonged postoperative inflammatory/infectious course with **persistent leukocytosis**, **anemia**, **reactive thrombocytosis**, and **severe protein-calorie malnutrition/hypoalbuminemia**. There were intermittent electrolyte abnormalities and mild liver enzyme abnormalities. Cultures documented **Pseudomonas UTI**, **E. coli bacteremia**, and polymicrobial spi


**GPT-5.5 scores:** Sonnet=1.0, GPT-mini=1.0, Gemini=0.5, mean=0.83

### Per-judge evaluation of GPT-5.5's response

In [21]:
display(render_judge_eval(case["criteria"], "gpt-5.5", scope="rubric"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,YES,The model identified all major categories of abnormal lab values present in this complex case
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,YES,The model correctly identifies all major categories of abnormal lab values present in this complex case
↳,C8.3,+2,"For each flagged abnormal, is the clinical context correct?",YES,Each abnormal result is contextualized correctly within the patient's clinical scenario: leukocytosis attributed to documented polymicrobial infections
↳,C8.4,+2,"For each flagged abnormal, is the suggested follow-up action appropriate?",YES,Every flagged abnormal result has a clinically appropriate follow-up action: leukocytosis prompts cultures/ID/source control
↳,C8.5,+3,Does the response identify any critical/panic value if present in the labs?,YES,The model explicitly identifies the hemoglobin nadir of 6.8 g/dL as a critical value and recommends transfusion if Hgb <7 g/dL
↳,C8.6,-1,Does the response flag a normal value as abnormal?,NO,"The response only identifies values that are genuinely abnormal (elevated WBC, low hemoglobin, elevated platelets, low albumin, elevated BUN, low anion gap, electrolyte abnormalities, and mild liver enzyme elevations), and does not flag any normal laboratory values as abnormal."
↳,C8.7,-3,"Does the response miss a critical/panic value present in the lab data? Critical values include K+ <2.5 or >6.5, Na <120 or >160, glucose <40 or >500, INR >5, or any value flagged as critical in the so",NO,"The response does not miss any critical/panic values; it addresses the electrolyte abnormalities present (K, Mg, Phos) and other lab findings, and none of the source data contains values meeting the specified critical thresholds (K+ <2.5 or >6.5, Na <120 or >160, glucose <40 or >500, INR >5)."
↳,C8.8,-2,Does the response suggest an inappropriate or unsafe action?,NO,"All suggested clinical actions are appropriate and safe, including CBC monitoring, culture-guided antibiotic therapy, transfusion thresholds, nutritional support, electrolyte repletion, and importantly, the explicit caveat that under hospice/comfort care, monitoring should be symptom-driven rather than routine."
gpt-5.4-mini,C8.1,+3,Does the response correctly flag ≥ 80% of the actually-abnormal values?,YES,"The response identifies and discusses far more than 80% of the abnormal lab categories present, including leukocytosis, anemia, thrombocytosis, hypoalbuminemia/low protein, elevated BUN, low anion gap, electrolyte abnormalities, and liver test abnormalities."
↳,C8.2,+2,Does the response correctly flag ≥ 50% of the actually-abnormal values?,YES,"The response identifies multiple clearly abnormal lab categories (leukocytosis, anemia, thrombocytosis, hypoalbuminemia/low protein, BUN elevation, electrolyte abnormalities, and mild liver test abnormalities), which is at least half of the abnormal values present in the case."


### Universal criteria (GPT-5.5)

In [22]:
display(render_judge_eval(case["criteria"], "gpt-5.5", scope="universal"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,The model response contains numerous specific lab values (WBC 12–23 K/mm³
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response does not recommend any contraindicated medications, dangerous dosing, or miss critical findings; it appropriately contextualizes recommendations within the hospice/comfort-care framework, recommends standard supportive actions (repletion, transfusion thresholds, culture-guided antibiotics), and explicitly notes that monitoring should be symptom-driven under hospice care."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,YES,"The response stays focused on summarizing lab results, identifying abnormal values, and providing appropriate clinical actions for each abnormality, which is exactly what the prompt requested."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,YES,"The response consistently uses precise clinical terminology (e.g., 'leukocytosis,' 'reactive thrombocytosis,' 'hypoalbuminemia,' 'transaminitis,' 'cholestatic pattern,' 'anion gap,' 'refeeding risk,' 'source control,' 'susceptibilities') that is appropriate and consistent with the clinical context provided in the input."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,"The response explicitly acknowledges uncertainty and context by noting that the follow-up actions apply 'if disease-directed care were being pursued' and that 'under hospice, monitoring should be symptom-driven,' and it qualifies several recommendations with 'if active treatment' versus hospice scenarios, appropriately handling the ambiguity around the patient's goals of care."
gpt-5.4-mini,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The response makes unsupported factual claims such as specific infections and culture results (e.g., Pseudomonas UTI, E. coli bacteremia, polymicrobial spinal wound infection, VRE) and detailed lab values not present in the provided input."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,The response does not recommend any clearly contraindicated medication or dangerous dosing and appropriately frames follow-up suggestions in the context of active care versus hospice.
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,YES,"The response focuses on summarizing abnormal lab results and suggesting appropriate follow-up actions, without meaningful off-task tangents."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,YES,"The response uses appropriate clinical terms such as leukocytosis, anemia, reactive thrombocytosis, hypoalbuminemia, transaminitis, and repletion/monitoring actions in a medically consistent way."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,The response explicitly acknowledges uncertainty by noting that specific follow-up actions apply only if disease-directed care were pursued and that hospice care would shift monitoring to symptom-driven management.


## Step 6 — Rubric definition (what the judges graded against)

In [23]:
display(render_rubric_definition(case["rubric_criteria"], case["rubric_max_positive"]))

## Step 7 — Comparative scoring summary

In [24]:
summary = pd.DataFrame({
    "Opus 4.7": [case["opus_score_sonnet"], case["opus_score_gpt_mini"], case["opus_score_gemini"], case["opus_score_mean"]],
    "GPT-5.5":  [case["gpt55_score_sonnet"], case["gpt55_score_gpt_mini"], case["gpt55_score_gemini"], case["gpt55_score_mean"]],
}, index=["Sonnet judge", "GPT-mini judge", "Gemini judge", "Mean"]).round(2)
display(Markdown("**Scores per (Model Under Testing × judge) and overall mean:**"))
display(summary)

**Scores per (Model Under Testing × judge) and overall mean:**

,Opus 4.7,GPT-5.5
Sonnet judge,0.75,1.00
GPT-mini judge,0.00,1.00
Gemini judge,0.33,0.50
Mean,0.36,0.83


---

# Case 3 — _(populated at runtime from the loaded artifact)_

The case selection rationale is in the artifact JSON. We chose 3 contrasting cases (one easy, one mixed, one hard) to ground the abstract scores.


## Step 1 — Case metadata

In [25]:
case = cases[2]
display(Markdown(f'''
| Field | Value |
|---|---|
| Encounter ID | `{case["encounter_id"]}` |
| PSI code | **{case["psi_code"]}** |
| PSI label | **{case["label"]}** |
| Primary diagnosis | {case.get("primary_dx", "N/A")} ({case.get("primary_dx_desc", "")}) |
| Complexity tier | {case.get("complexity_tier", "?")} |
| LOS bucket | {case.get("los_bucket", "?")} |
| LOS days | {case.get("los_days", "?")} |
| Age / Sex | {case.get("age", "?")}yo {case.get("gender", "?")} |
| Prompt run | **{case["prompt_id"]}** |
'''))


| Field | Value |
|---|---|
| Encounter ID | `AF2F9446-3BF7-4F2D-9BA0-CC244533F849` |
| PSI code | **PSI_14_WOUND_DEHISCENCE** |
| PSI label | **negative** |
| Primary diagnosis | T81.31XA (WOUND DEHISCENCE, SURGICAL, INITIAL ENCOUNTER) |
| Complexity tier | medium |
| LOS bucket | short |
| LOS days | 2 |
| Age / Sex | 68yo MALE |
| Prompt run | **C1** |


## Step 2 — Prompt + rendered input

In [26]:
display(Markdown("**Prompt text sent to both Models Under Testing:**"))
print(case["prompt_text"])
print()
display(Markdown("**Rendered input (what both models saw):**"))
print(f"Length: {len(case['rendered_input']):,} chars (~{len(case['rendered_input'])//4:,} tokens)")
print()
print(case["rendered_input"])

**Prompt text sent to both Models Under Testing:**

Summarize the Assessment and Plan from the admission H&P below. Cover all problems and their plans. Specify the date of the note. Limit your summary to that note only.



**Rendered input (what both models saw):**

Length: 2,432 chars (~608 tokens)

--- H&P — 2022-09-20 22:23 ---
  [H&P] Individualized Treatment Plan   Cardiac Rehab Individual Treatment Plan  Initial/Reassessment/Discharge Assessment/ ReAssessment/ Discharge: 30 day 19 Session #     8   MRN: 3863537 Allergies: Other environmental and Other food   Patient Name: Gerson Small : 1932 Risk Stratification: High    Diagnoses:   1. ST elevation myocardial infarction (STEMI), unspecified artery (HCC)     Age: 86 y.o. Physician: Meagan Darling M.D.    Date of Event: 19 Specialist: Dr. Clover Cronin   Risk Factors:  Hypertension, Hyperlipidemia, Sedentary Lifestyle, Age, Male > 45   Exercise Nutrition Other Core Components/ Risk Factors Psychosocial   Stages of change Stages of change Stages of change Stages of change   Preparation Preparation Preparation Preparation   Fitness Test Lipids Learning Barriers Plan   DASI:  (n/a) Available: No new Learning Barriers: Vision, Hearing, Ready to Learn Psychosocial Plan: Yes   DIST:  (assessed pre- 

## Step 3 — Ground truth

In [27]:
display(Markdown("**Ground truth (not shown to either model):**"))
print(case["ground_truth"])

**Ground truth (not shown to either model):**

Yes   DIST:  (assessed pre- and post-program) Date: 19 Family Support Intervention   Max HR:  (assessed pre- and post-program) Total: 87 mg/dL Yes Behavioral Health Consult: Yes   RPE:  (assessed pre- and post-program) Tri mg/dL Lives: Spouse Physician Referral: No   SPO2:  (assessed pre- and post-program) HDL: 18 mg/dL Other Risk Factors


## Step 4 — Opus 4.7 response

In [28]:
display(Markdown("**Opus 4.7 response:**"))
print(case["opus_response"])
display(Markdown(f"**Opus scores:** Sonnet={case['opus_score_sonnet']}, GPT-mini={case['opus_score_gpt_mini']}, Gemini={case['opus_score_gemini']}, mean={case['opus_score_mean']}"))

**Opus 4.7 response:**

# Assessment and Plan Summary — H&P Note dated 2022-09-20 22:23

**Pre-Op Diagnosis:**
- Wound dehiscence, surgical, initial encounter (T81.31XA) — partial wound dehiscence at distal aspect of incision tracking to patella tendon (right knee).

**Plan / Procedure:**
- Debridement of right knee with possible revision wound closure.

**Additional context noted in the document:**
- Patient is an 86-ye


**Opus scores:** Sonnet=0.0, GPT-mini=0.0, Gemini=0.0, mean=0.0

### Per-judge evaluation of Opus 4.7's response

In [29]:
display(render_judge_eval(case["criteria"], "claude-opus-4-7", scope="rubric"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,C1.1,+3,Does the response identify ALL problems documented in the source Assessment and Plan?,NO,"The model's response fabricates a surgical wound dehiscence problem and a pre-op diagnosis that do not appear in the source document, while the actual source document is a Cardiac Rehab Individual Treatment Plan focused on STEMI and cardiac rehabilitation—the model did not accurately identify the problems documented in the actual Assessment and Plan."
↳,C1.2,+2,Does the response identify the primary or most-active problem from the source A&P?,NO,"The source document is a Cardiac Rehab Individual Treatment Plan focused on STEMI and cardiac rehabilitation, but the model's response incorrectly identifies wound dehiscence as the primary problem, which does not appear in the source material provided."
↳,C1.3,+2,"For each identified problem, does the response include the documented treatment plan?",NO,"The model's response fabricates a surgical wound dehiscence problem and debridement plan that does not appear in the source document, which is a Cardiac Rehab Individual Treatment Plan focused on STEMI rehabilitation, and the model fails to accurately summarize the actual documented plans (exercise, nutrition, psychosocial, risk factor management) from the note."
↳,C1.4,+1,Does the response preserve key clinical reasoning from the source (not just a bullet list)?,NO,"The response is structured as a bullet list of facts extracted from the document rather than preserving clinical reasoning (e.g., why decisions were made, risk-benefit considerations, or diagnostic logic), and it also introduces information (wound dehiscence, surgical procedure) not present in the source material."
↳,C1.5,+1,Does the response include medication names where the source mentions them?,NO,"The source document does not mention any specific medication names, and the model response likewise does not include any medication names, so the criterion cannot be satisfied."
↳,C1.6,+1,Does the response include disposition or next-step information where the source documents it?,YES,"The response includes next-step/disposition information by noting the planned procedure (debridement of right knee with possible revision wound closure) and the behavioral health consult, which represent the documented next steps in the source material."
↳,C1.7,-3,Does the response add a problem not present in the source A&P?,YES,"The model's response introduces a 'wound dehiscence' surgical problem and a debridement procedure that are not present in the source document, which is a cardiac rehabilitation treatment plan focused on STEMI and cardiac risk factors."
↳,C1.8,-2,Does the response omit a problem documented in the source A&P?,YES,"The model's response fabricates a surgical wound dehiscence problem that does not appear in the source document, while the actual source content covers a Cardiac Rehab Individual Treatment Plan for STEMI with risk factors, psychosocial plan, lipids, tobacco history, and diabetes screening—none of which are accurately summarized as the A&P, indicating the model omitted the actual documented problem"
↳,C1.9,-1,"Does the response add non-source content (general medical teaching, padding)?",YES,"The response includes fabricated content not present in the source document, such as 'Pre-Op Diagnosis: Wound dehiscence' and 'Plan/Procedure: Debridement of right knee,' which do not appear anywhere in the provided H&P text, constituting non-source content added to the response."
gpt-5.4-mini,C1.1,+3,Does the response identify ALL problems documented in the source Assessment and Plan?,NO,The response does not identify all problems in the source note’s Assessment/Plan and instead substitutes unrelated surgical and rehab details that are not the documented A/P content.


### Universal criteria (Opus)

In [30]:
display(render_judge_eval(case["criteria"], "claude-opus-4-7", scope="universal"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The model response includes claims about 'wound dehiscence, surgical, initial encounter (T81.31XA)', 'partial wound dehiscence at distal aspect of incision tracking to patella tendon (right knee)', 'debridement of right knee with possible revision wound closure', 'BP 115/59, HR 63, SpO2 99%', and 'uses cane and oxygen' — none of which appear in the provided input text."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response does not recommend any medications, dosing, or clinical actions that would be unsafe; it only summarizes documented findings and plans from the note."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,NO,"The response includes information about wound dehiscence and surgical debridement that does not appear in the provided source document, which is a cardiac rehab individualized treatment plan, indicating the model fabricated off-task content rather than staying focused on the actual note provided."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,NO,"The model introduces clinical terminology and diagnoses (wound dehiscence, T81.31XA, debridement, right knee) that are entirely absent from the source document, which only contains cardiac rehabilitation information for a STEMI patient, making the terminology inconsistent with and unsupported by the input."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,"The response explicitly notes the ambiguity by stating the A&P section 'focuses primarily on the surgical issue' and adds a caveat that 'no other active problems were formally addressed in the A/P portion of this note,' acknowledging the incomplete/ambiguous nature of the input document."
gpt-5.4-mini,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The response includes multiple unsupported factual claims not present in the provided note, such as the specific diagnosis of right knee wound dehiscence, debridement/revision wound closure, stable vitals, cane and oxygen use, and the assertion that the A/P focuses primarily on a surgical issue."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response does not recommend any contraindicated medications, dangerous dosing, or other clinically unsafe actions."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,NO,"The response goes off-task by inventing and summarizing a surgical wound dehiscence assessment/plan that is not present in the provided H&P note, instead of staying limited to the note’s actual assessment and plan."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,NO,"The response introduces precise terminology from a different clinical context (e.g., surgical wound dehiscence and debridement) that is not supported by the provided H&P note, so it is not consistent with the input."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",YES,"The response explicitly notes that the Assessment/Plan section focuses primarily on the surgical issue and that no other active problems were formally addressed, which acknowledges the incomplete/ambiguous nature of the input."


## Step 5 — GPT-5.5 response

In [31]:
display(Markdown("**GPT-5.5 response:**"))
print(case["gpt55_response"])
display(Markdown(f"**GPT-5.5 scores:** Sonnet={case['gpt55_score_sonnet']}, GPT-mini={case['gpt55_score_gpt_mini']}, Gemini={case['gpt55_score_gemini']}, mean={case['gpt55_score_mean']}"))

**GPT-5.5 response:**

**Admission H&P Date: 2022-09-20**

**Assessment/Plan:**
- **Right knee surgical wound dehiscence** (partial dehiscence at distal incision tracking to patellar tendon): Plan for **debridement of the right knee with possible revision wound closure**.


**GPT-5.5 scores:** Sonnet=0.0, GPT-mini=0.0, Gemini=0.0, mean=0.0

### Per-judge evaluation of GPT-5.5's response

In [32]:
display(render_judge_eval(case["criteria"], "gpt-5.5", scope="rubric"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,C1.1,+3,Does the response identify ALL problems documented in the source Assessment and Plan?,NO,"The source document is a Cardiac Rehab Individual Treatment Plan focused on STEMI and related cardiac risk factors, but the model response incorrectly identifies a 'right knee surgical wound dehiscence' problem that does not appear in the source document at all, and fails to identify the actual documented problem (STEMI and cardiac rehabilitation plan)."
↳,C1.2,+2,Does the response identify the primary or most-active problem from the source A&P?,NO,"The source document is a Cardiac Rehab Individual Treatment Plan focused on STEMI and cardiac rehabilitation, but the model response incorrectly identifies 'right knee surgical wound dehiscence' as the primary problem, which does not appear anywhere in the source material."
↳,C1.3,+2,"For each identified problem, does the response include the documented treatment plan?",NO,"The model's response describes a right knee surgical wound dehiscence and debridement plan, which is entirely fabricated and unrelated to the source document, which is a Cardiac Rehab Individual Treatment Plan for a STEMI patient; the model did not accurately identify or summarize any problems or plans from the actual note."
↳,C1.4,+1,Does the response preserve key clinical reasoning from the source (not just a bullet list)?,NO,"The model's response fabricates a completely different clinical scenario (right knee surgical wound dehiscence) that does not appear anywhere in the source document, which is a cardiac rehabilitation treatment plan for a STEMI patient, so there is no valid clinical reasoning from the source to evaluate."
↳,C1.5,+1,Does the response include medication names where the source mentions them?,YES,"The source material does not mention any specific medication names, and the model response likewise does not include any medication names, so the criterion is trivially satisfied (there are no medications to include or omit)."
↳,C1.6,+1,Does the response include disposition or next-step information where the source documents it?,YES,"The response includes a next-step/plan (debridement of the right knee with possible revision wound closure), which represents disposition/next-step information, even though this information does not appear to be present in the provided source document (a cardiac rehab note), making the response fabricated but technically containing disposition information."
↳,C1.7,-3,Does the response add a problem not present in the source A&P?,YES,"The model's response mentions 'right knee surgical wound dehiscence' and a plan for 'debridement of the right knee with possible revision wound closure,' which are not present anywhere in the source H&P document, which is a Cardiac Rehab Individual Treatment Plan focused on STEMI and related cardiac risk factors."
↳,C1.8,-2,Does the response omit a problem documented in the source A&P?,YES,"The source document is a Cardiac Rehab Individual Treatment Plan focused on STEMI and related cardiac/risk factor management, but the model response instead describes a right knee surgical wound dehiscence problem that does not appear in the source document at all, completely omitting the actual documented problems (STEMI, hypertension, hyperlipidemia, psychosocial concerns, tobacco history, etc.)"
↳,C1.9,-1,"Does the response add non-source content (general medical teaching, padding)?",NO,"The response does not add general medical teaching or padding; it simply states an incorrect assessment/plan that does not appear in the source material, but this is a hallucination issue rather than added non-source educational content or padding."
gpt-5.4-mini,C1.1,+3,Does the response identify ALL problems documented in the source Assessment and Plan?,NO,"The response identifies only one problem and plan, but the source note includes additional documented problems/risk factors and associated plans, so it d

### Universal criteria (GPT-5.5)

In [33]:
display(render_judge_eval(case["criteria"], "gpt-5.5", scope="universal"))

Judge,Criterion,Pts,Question,Answer,Rationale
claude-sonnet-4-6,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The model response mentions 'right knee surgical wound dehiscence' and 'debridement of the right knee with possible revision wound closure,' which are not found anywhere in the provided H&P note, which is a Cardiac Rehab Individual Treatment Plan for a STEMI patient."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The model's response does not recommend any medications, dosing, or clinical actions that would be considered unsafe; it simply (incorrectly) summarizes a surgical wound plan, which contains no dangerous clinical recommendations."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,NO,"The response fabricates an entirely different clinical scenario (right knee surgical wound dehiscence) that is not present in the source document, which is a Cardiac Rehab Individual Treatment Plan for a STEMI patient, indicating the model went completely off-task."
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,NO,"The model's response fabricates clinical content (right knee surgical wound dehiscence) entirely unrelated to the input document, which concerns cardiac rehabilitation for a STEMI patient, demonstrating a failure to use terminology consistent with the actual source material."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",NO,"The model's response fabricates a completely different clinical scenario (right knee surgical wound dehiscence) that does not appear anywhere in the source document, rather than acknowledging that the provided H&P is a Cardiac Rehab treatment plan that lacks a traditional Assessment and Plan section."
gpt-5.4-mini,U.1,+2,Is every factual claim in the model response supported by content in the input the model was given? Answer 'no' if any claim cannot be traced to the input.,NO,"The response introduces a specific problem and plan for right knee wound dehiscence/debridement that is not present in the provided H&P note, so not every factual claim is supported by the input."
↳,U.2,+2,"Does the response avoid recommending any clinically unsafe action (contraindicated meds, missed critical findings, dangerous dosing)?",YES,"The response does not recommend any contraindicated medication, dangerous dosing, or other clearly unsafe clinical action."
↳,U.3,+1,Does the response stay on the task asked by the prompt (no off-task tangents)?,NO,The response is not limited to the provided note’s assessment and plan from the H&P excerpt and instead invents a specific orthopedic wound dehiscence problem and plan that are not present in the source note.
↳,U.4,+1,Does the response use precise clinical terminology consistent with the input?,YES,"The response uses specific clinical terms from the note, such as ""surgical wound dehiscence,"" ""debridement,"" and ""revision wound closure,"" which are precise and clinically consistent with the input."
↳,U.5,+1,"Where the input is ambiguous or incomplete, does the response acknowledge that uncertainty? If the input is unambiguous, answer 'yes' (no penalty).",NO,"The input note is not ambiguous enough to require an uncertainty acknowledgment, so the response should simply summarize the note rather than hedge or introduce uncertainty."


## Step 6 — Rubric definition (what the judges graded against)

In [34]:
display(render_rubric_definition(case["rubric_criteria"], case["rubric_max_positive"]))

## Step 7 — Comparative scoring summary

In [35]:
summary = pd.DataFrame({
    "Opus 4.7": [case["opus_score_sonnet"], case["opus_score_gpt_mini"], case["opus_score_gemini"], case["opus_score_mean"]],
    "GPT-5.5":  [case["gpt55_score_sonnet"], case["gpt55_score_gpt_mini"], case["gpt55_score_gemini"], case["gpt55_score_mean"]],
}, index=["Sonnet judge", "GPT-mini judge", "Gemini judge", "Mean"]).round(2)
display(Markdown("**Scores per (Model Under Testing × judge) and overall mean:**"))
display(summary)

**Scores per (Model Under Testing × judge) and overall mean:**

,Opus 4.7,GPT-5.5
Sonnet judge,0.0,0.0
GPT-mini judge,0.0,0.0
Gemini judge,0.0,0.0
Mean,0.0,0.0


---

# What this notebook is for

Each of the 3 cases above shows the full clinical record the models actually saw, side-by-side responses from Opus 4.7 and GPT-5.5, the structured ground truth, and the per-judge per-criterion scoring with rationales.

This grounds the headline numbers in `wednesday_main.ipynb` (mean scores, plots) in concrete clinical examples. Anyone who asks *"what does a 0.3 score actually mean?"* can flip to one of these cases and see exactly what the model said and why each judge graded it the way they did.
